# ETL Silver → Gold
## Crime Data Pipeline

Pipeline de transformação para criação do modelo dimensional.

**Objetivo**: Criar modelo Star Schema na camada Gold para analytics e BI.

**Entrada**: PostgreSQL (`silver.crimes`)  
**Saída**: PostgreSQL (`gold.*`)

In [ ]:
# Configuração inicial
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os
import sys
from psycopg2.extras import execute_values

# Configurar caminhos
PROJECT_ROOT = Path.cwd().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from postgres.helpers.postgres_helper import pg_helper
from postgres.plugins.cliente_postgres import CrimeDataPostgresClient

client = CrimeDataPostgresClient()

print(f"Projeto: {PROJECT_ROOT}")
print("Silver: PostgreSQL (schema silver, tabela crimes)")
print("Gold: PostgreSQL (schema gold)")

In [ ]:
# Carregar dados Silver do PostgreSQL
query = """
SELECT
    crime_id,
    date_occurred,
    date_reported,
    time_occurred,
    hour_occurred,
    area_code,
    area_name,
    district_number,
    crime_code,
    crime_description,
    crime_part,
    crime_category,
    victim_age,
    victim_sex,
    victim_descent,
    descent_description,
    premise_code,
    premise_description,
    weapon_code,
    weapon_description,
    status_code,
    status_description,
    latitude,
    longitude,
    location,
    day_of_week,
    day_name,
    is_weekend,
    period_of_day,
    is_violent,
    year_occurred,
    month_occurred,
    age_group
FROM silver.crimes
"""

df_silver = pg_helper.execute_query_df(query)

# Ajustes de tipos
df_silver['date_occurred'] = pd.to_datetime(df_silver['date_occurred'], errors='coerce')
df_silver['date_reported'] = pd.to_datetime(df_silver['date_reported'], errors='coerce')
for col in ['area_code', 'crime_code', 'weapon_code', 'premise_code', 'crime_part', 'victim_age']:
    df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

# Campos derivados para compatibilidade com o pipeline
df_silver['hour'] = pd.to_numeric(df_silver['hour_occurred'], errors='coerce')

sex_map = {'M': 'Male', 'F': 'Female', 'X': 'Unknown', 'H': 'Unknown', '-': 'Unknown'}
df_silver['victim_sex_desc'] = df_silver['victim_sex'].map(sex_map).fillna('Unknown')
df_silver['victim_descent_desc'] = df_silver['descent_description'].fillna('Unknown')
df_silver['victim_age_group'] = df_silver['age_group'].fillna('Unknown')
df_silver['crime_severity'] = df_silver['crime_part'].map({1: 'Serious', 2: 'Minor'})
df_silver['has_weapon'] = (df_silver['weapon_code'].fillna(0).astype(int) != 0) | (df_silver['weapon_description'].fillna('').str.strip() != '')
df_silver['case_closed'] = df_silver['status_code'].isin(['AA', 'JA'])

print(f"Dados Silver carregados: {len(df_silver):,} registros")
print(f"Colunas: {len(df_silver.columns)}")
df_silver.head(3)

In [ ]:
# Validações padronizadas de schema e qualidade (Silver)
print("Validando schema e qualidade...")

required_cols = [
    'crime_id', 'date_occurred', 'date_reported', 'hour',
    'area_code', 'area_name',
    'crime_code', 'crime_description', 'crime_category', 'crime_severity',
    'victim_age_group', 'victim_sex_desc', 'victim_descent_desc',
    'victim_age',
    'latitude', 'longitude',
    'is_violent', 'has_weapon', 'case_closed',
    'year', 'month'
 ]

null_thresholds = {
    'crime_id': 0.00,
    'date_occurred': 0.01,
    'hour': 0.01,
    'area_code': 0.01,
    'crime_code': 0.01
}

def validate_silver_schema(df):
    errors = []
    warnings = []

    if df.empty:
        errors.append("Dataset vazio.")

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        errors.append(f"Colunas ausentes: {missing}")

    for col in ['date_occurred', 'date_reported']:
        if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
            warnings.append(f"{col} não está em datetime64; verifique conversão.")

    for col, max_null in null_thresholds.items():
        if col in df.columns:
            pct = df[col].isna().mean()
            if pct > max_null:
                errors.append(f"{col} com {pct:.1%} nulos (limite {max_null:.1%}).")

    if 'hour' in df.columns:
        invalid = ~df['hour'].between(0, 23)
        if invalid.any():
            errors.append(f"hour fora de 0-23: {invalid.sum():,} registros.")

    if 'latitude' in df.columns and 'longitude' in df.columns:
        coords = df[['latitude', 'longitude']].dropna()
        if not coords.empty:
            la_bounds = coords['latitude'].between(33.7, 34.4) & coords['longitude'].between(-118.7, -118.1)
            outside = (~la_bounds).sum()
            if outside / len(coords) > 0.05:
                warnings.append(f"{outside:,} coordenadas fora do limite LA (>5%).")

    if 'crime_severity' in df.columns:
        allowed = {'Serious', 'Minor'}
        invalid = ~df['crime_severity'].dropna().isin(allowed)
        if invalid.any():
            warnings.append(f"crime_severity fora do domínio esperado: {invalid.sum():,} registros.")

    if 'crime_id' in df.columns:
        dup = df['crime_id'].duplicated().sum()
        if dup > 0:
            warnings.append(f"crime_id duplicado: {dup:,}")

    return errors, warnings

errors, warnings = validate_silver_schema(df_silver)
if warnings:
    print("Avisos:")
    for w in warnings:
        print(f"   - {w}")

if errors:
    print("Erros:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("Falha nas validações de schema/qualidade. Corrija antes de gerar a Gold.")
else:
    print("Validações concluídas com sucesso.")

In [ ]:
## Criação das Dimensões

In [ ]:
# Dimensão: Data (dim_date)
print("Criando dim_date...")

dim_date = df_silver[['date_occurred']].drop_duplicates().copy()
dim_date = dim_date.dropna()
dim_date = dim_date.rename(columns={'date_occurred': 'full_date'})
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['week_of_year'] = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['day_of_month'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['day_of_week'].isin([5, 6])
dim_date['is_holiday'] = False

count = client.load_dimension(dim_date, table='dim_date', key_column='full_date')
print(f"   dim_date: {count:,} registros")

In [ ]:
# Dimensão: Tempo (dim_time)
print("Criando dim_time...")

dim_time = pd.DataFrame({'hour': range(24)})
dim_time['minute'] = 0
dim_time['full_time'] = pd.to_datetime(dim_time['hour'].astype(str).str.zfill(2) + ':00', format='%H:%M').dt.time
dim_time['period_of_day'] = dim_time['hour'].apply(
    lambda h: 'Madrugada' if h < 6 else 'Manhã' if h < 12 else 'Tarde' if h < 18 else 'Noite'
 )
dim_time['is_rush_hour'] = dim_time['hour'].isin([7, 8, 9, 17, 18, 19])

count = client.load_dimension(dim_time, table='dim_time', key_column='full_time')
print(f"   dim_time: {count:,} registros")

In [ ]:
# Dimensão: Área (dim_area)
print("Criando dim_area...")

dim_area = df_silver[['area_code', 'area_name']].drop_duplicates().copy()

# Classificar regiões
def get_region(area_name):
    north = ['DEVONSHIRE', 'FOOTHILL', 'MISSION', 'NORTH HOLLYWOOD', 'VAN NUYS', 'WEST VALLEY']
    south = ['77TH STREET', 'HARBOR', 'SOUTHEAST', 'SOUTHWEST']
    central = ['CENTRAL', 'HOLLENBECK', 'RAMPART']
    west = ['HOLLYWOOD', 'OLYMPIC', 'PACIFIC', 'WEST LA', 'WILSHIRE']
    
    if area_name in north: return 'North'
    elif area_name in south: return 'South'
    elif area_name in central: return 'Central'
    elif area_name in west: return 'West'
    else: return 'Other'

dim_area['region'] = dim_area['area_name'].apply(get_region)

count = client.load_dimension(dim_area, table='dim_area', key_column='area_code')
print(f"   dim_area: {count:,} registros")

In [ ]:
# Dimensão: Tipo de Crime (dim_crime_type)
print("Criando dim_crime_type...")

dim_crime_type = df_silver[['crime_code', 'crime_description', 'crime_category', 'crime_severity']].drop_duplicates().copy()
dim_crime_type['is_violent'] = dim_crime_type['crime_category'] == 'Violent Crime'
dim_crime_type['severity_level'] = dim_crime_type['crime_severity'].map({'Serious': 3, 'Minor': 1})

count = client.load_dimension(dim_crime_type, table='dim_crime_type', key_column='crime_code')
print(f"   dim_crime_type: {count:,} registros")

In [ ]:
# Dimensões: Arma e Premissa (dim_weapon, dim_premise)
print("Criando dim_weapon e dim_premise...")

def get_weapon_category(desc):
    desc = str(desc).upper() if pd.notna(desc) else ''
    if 'GUN' in desc or 'FIREARM' in desc or 'RIFLE' in desc or 'REVOLVER' in desc:
        return 'Firearm'
    if 'KNIFE' in desc or 'BLADE' in desc or 'CUTTING' in desc:
        return 'Blade'
    if 'BLUNT' in desc or 'CLUB' in desc or 'BAT' in desc:
        return 'Blunt Object'
    if 'STRONG-ARM' in desc or 'HANDS' in desc or 'FIST' in desc:
        return 'Physical Force'
    if desc == '' or desc == 'NAN':
        return 'No Weapon'
    return 'Other Weapon'

def get_premise_category(desc):
    desc = str(desc).upper() if pd.notna(desc) else ''
    if any(x in desc for x in ['STREET', 'SIDEWALK', 'ALLEY', 'PARKING', 'PARK', 'BEACH']):
        return 'Public'
    if any(x in desc for x in ['DWELLING', 'HOUSE', 'APARTMENT', 'RESIDENCE', 'CONDOMINIUM']):
        return 'Residential'
    if any(x in desc for x in ['STORE', 'SHOP', 'RESTAURANT', 'BAR', 'BANK', 'HOTEL', 'MARKET']):
        return 'Commercial'
    return 'Other'

# dim_weapon
dim_weapon = df_silver[['weapon_code', 'weapon_description']].drop_duplicates().copy()
dim_weapon = dim_weapon[dim_weapon['weapon_code'].notna()]
dim_weapon['weapon_category'] = dim_weapon['weapon_description'].apply(get_weapon_category)
dim_weapon['lethality_level'] = dim_weapon['weapon_category'].map({
    'Firearm': 5,
    'Blade': 3,
    'Blunt Object': 2,
    'Physical Force': 1,
    'No Weapon': 0,
    'Other Weapon': 1
})

weapon_count = client.load_dimension(dim_weapon, table='dim_weapon', key_column='weapon_code')
print(f"   dim_weapon: {weapon_count:,} registros")

# dim_premise
dim_premise = df_silver[['premise_code', 'premise_description']].drop_duplicates().copy()
dim_premise = dim_premise[dim_premise['premise_code'].notna()]
dim_premise['premise_category'] = dim_premise['premise_description'].apply(get_premise_category)
dim_premise['is_public'] = dim_premise['premise_category'].eq('Public')

premise_count = client.load_dimension(dim_premise, table='dim_premise', key_column='premise_code')
print(f"   dim_premise: {premise_count:,} registros")

In [ ]:
# Dimensão: Vítima (dim_victim)
print("Criando dim_victim...")

dim_victim = df_silver[['victim_age_group', 'victim_sex', 'victim_descent', 'victim_descent_desc']].drop_duplicates().copy()
dim_victim = dim_victim.rename(columns={
    'victim_age_group': 'age_group',
    'victim_sex': 'sex',
    'victim_descent': 'descent',
    'victim_descent_desc': 'descent_description'
})

values = [tuple(row) for row in dim_victim.values]
upsert_query = """
    INSERT INTO gold.dim_victim (age_group, sex, descent, descent_description)
    VALUES %s
    ON CONFLICT (age_group, sex, descent) DO UPDATE SET
        descent_description = EXCLUDED.descent_description
"""
with pg_helper.get_cursor() as cursor:
    execute_values(cursor, upsert_query, values, page_size=1000)

print(f"   dim_victim: {len(dim_victim):,} registros")

In [ ]:
## Criação da Tabela Fato

In [ ]:
# Tabela Fato: fato_crimes
print("Criando fato_crimes...")

# Mapeamentos de surrogate keys
date_map = client.get_dimension_keys('dim_date', 'full_date', 'sk_date')
time_map = client.get_dimension_keys('dim_time', 'full_time', 'sk_time')
area_map = client.get_dimension_keys('dim_area', 'area_code', 'sk_area')
crime_type_map = client.get_dimension_keys('dim_crime_type', 'crime_code', 'sk_crime_type')
weapon_map = client.get_dimension_keys('dim_weapon', 'weapon_code', 'sk_weapon')
premise_map = client.get_dimension_keys('dim_premise', 'premise_code', 'sk_premise')

dim_victim_db = pg_helper.execute_query_df("SELECT sk_victim, age_group, sex, descent FROM gold.dim_victim")
dim_victim_db['victim_key'] = dim_victim_db['age_group'].astype(str) + '|' + dim_victim_db['sex'].astype(str) + '|' + dim_victim_db['descent'].astype(str)
victim_map = dict(zip(dim_victim_db['victim_key'], dim_victim_db['sk_victim']))

# Criar chave composta para vítima
df_silver['victim_key'] = df_silver['victim_age_group'].astype(str) + '|' + df_silver['victim_sex'].astype(str) + '|' + df_silver['victim_descent'].astype(str)

# Chaves de data/tempo
df_silver['date_key'] = df_silver['date_occurred'].dt.date
df_silver['full_time'] = pd.to_datetime(df_silver['hour'].fillna(0).astype(int).astype(str).str.zfill(2) + ':00', format='%H:%M').dt.time

# Construir fato
fato = pd.DataFrame()
fato['nk_crime_id'] = df_silver['crime_id'].values
fato['sk_area'] = df_silver['area_code'].map(area_map).values
fato['sk_crime_type'] = df_silver['crime_code'].map(crime_type_map).values
fato['sk_weapon'] = df_silver['weapon_code'].map(weapon_map).values
fato['sk_premise'] = df_silver['premise_code'].map(premise_map).values
fato['sk_date'] = df_silver['date_key'].map(date_map).values
fato['sk_time'] = df_silver['full_time'].map(time_map).values
fato['sk_victim'] = df_silver['victim_key'].map(victim_map).values
fato['latitude'] = df_silver['latitude'].values
fato['longitude'] = df_silver['longitude'].values
fato['is_violent'] = df_silver['is_violent'].values

pg_helper.truncate_table('fato_crimes', schema='gold')
count = client.load_fact(fato, table='fato_crimes')
print(f"   fato_crimes: {count:,} registros")

In [ ]:
## Criação das Agregações

In [ ]:
# Agregações na camada Gold
print("Criando agregações...")

# Agregação: Crimes por Área e Período
agg_area_period = df_silver.copy()
agg_area_period['sk_area'] = df_silver['area_code'].map(area_map)
agg_area_period['year'] = df_silver['date_occurred'].dt.year
agg_area_period['month'] = df_silver['date_occurred'].dt.month
agg_area_period['period_of_day'] = df_silver['period_of_day']
agg_area_period['is_property'] = df_silver['crime_category'].eq('Property Crime')

agg_area_period = agg_area_period.groupby(['sk_area', 'year', 'month', 'period_of_day']).agg(
    total_crimes=('crime_id', 'count'),
    violent_crimes=('is_violent', 'sum'),
    property_crimes=('is_property', 'sum'),
    avg_victim_age=('victim_age', 'mean')
).reset_index()

pg_helper.truncate_table('agg_crimes_area_period', schema='gold')
pg_helper.insert_dataframe(agg_area_period, table='agg_crimes_area_period', schema='gold')
print(f"   agg_crimes_area_period: {len(agg_area_period):,} registros")

# Agregação: Crimes por Tipo e Ano
agg_crime_year = df_silver.copy()
agg_crime_year['sk_crime_type'] = df_silver['crime_code'].map(crime_type_map)
agg_crime_year['year'] = df_silver['date_occurred'].dt.year
agg_crime_year['is_weekend'] = df_silver['is_weekend']
agg_crime_year['is_weekday'] = ~agg_crime_year['is_weekend']

agg_crime_year = agg_crime_year.groupby(['sk_crime_type', 'year']).agg(
    total_crimes=('crime_id', 'count'),
    weekday_crimes=('is_weekday', 'sum'),
    weekend_crimes=('is_weekend', 'sum')
).reset_index()

pg_helper.truncate_table('agg_crimes_type_year', schema='gold')
pg_helper.insert_dataframe(agg_crime_year, table='agg_crimes_type_year', schema='gold')
print(f"   agg_crimes_type_year: {len(agg_crime_year):,} registros")

In [ ]:
# Resumo final
print("\n" + "="*50)
print("ETL Silver → Gold concluído!")
print("="*50)

print("\nCarga concluída no PostgreSQL:")
print("   - gold.dim_date")
print("   - gold.dim_time")
print("   - gold.dim_area")
print("   - gold.dim_crime_type")
print("   - gold.dim_weapon")
print("   - gold.dim_premise")
print("   - gold.dim_victim")
print("   - gold.fato_crimes")
print("   - gold.agg_crimes_area_period")
print("   - gold.agg_crimes_type_year")